### Task 1 

Ask 10 different questions to your RAG chatbot. 

For each question: 


- Check whether the retrieved documents are relevant. 
- Check whether the final answer is correct. 

In [1]:
#Load All PDFs from the Folder
import os 

from langchain_community.document_loaders import PyPDFLoader 

documents = [] 
folder = "Documents" 

for file in os.listdir(folder): 

    if file.endswith(".pdf"): 

        loader = PyPDFLoader(os.path.join(folder, file)) 

        documents.extend(loader.load()) 

print(f"Loaded {len(documents)} pages.") 

C:\Users\sachi\AppData\Local\Temp\ipykernel_24504\1044489116.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\sachi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 33 pages.


In [2]:
#Splitting Multiple Documents 

from langchain_text_splitters import RecursiveCharacterTextSplitter 

splitter = RecursiveCharacterTextSplitter( 

    chunk_size=800, 

    chunk_overlap=150

) 
chunks = splitter.split_documents(documents) 

print(len(chunks)) 

99


In [3]:
#Load the embedding model
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3551.95it/s]


Embeddings model loaded successfully.


In [4]:
#Store Everything in ChromaDB
from langchain_chroma import Chroma 
vector_db = Chroma.from_documents( 

    documents=chunks, 

    embedding=embedding, 

    persist_directory="./chroma_db" 

) 

In [11]:
# Create a retriever for similarity search
retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20
    }
)
print("Vector database and retriever created successfully.")

Vector database and retriever created successfully.


In [ ]:
from langchain_ollama import OllamaLLM
#Load llm
llm = OllamaLLM(model="llama3.2")

# List of questions
questions = [
    "What is the eligibility criteria for CAT?",
    "When does CAT 2026 registration start?",
    "When does CAT 2026 registration close?",
    "Which IIM is conducting CAT 2026?",
    "What was the previous CAT cutoff for IIM Bangalore",
    "What are the mandatory documents required during registration?",
    "What is the duration of exam",
    "How many test cities can a candidate select during registration?",
    "When will CAT 2026 results be declared?",
    "What is the registration fee?"
]

# Loop through all questions
for question in questions:

    print("="*100)
    print("Question:")
    print(question)

    # Retrieve documents
    docs = retriever.invoke(question)

    print("\nRetrieved Documents:\n")

    context = ""

    for i, doc in enumerate(docs, start=1):

        print(f"----- Document {i} -----")
        print("Source:", doc.metadata["source"])
        print("Page:", doc.metadata["page"] + 1)
        print(doc.page_content)
        print("-"*60)

        context += doc.page_content + "\n\n"
    print(context)

    # Prompt
    prompt = f"""
You are a helpful assistant.

Answer ONLY using the information in the context below.

If the answer is not available in the context, reply:
"I don't know. The uploaded documents do not contain this information."

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer
    answer = llm.invoke(prompt)

    print("\nAnswer:\n")
    print(answer)
    print("\n")

Question:
What is the eligibility criteria for CAT?

Retrieved Documents:

----- Document 1 -----
Source: Documents\CAT_2026_Information_Bulletin_26-07-26.pdf
Page: 4
c) locomotor disability including cerebral palsy, leprosy cured, dwarfism, acid attack victims and 
muscular dystrophy,   
d) autism, intellectual disability, specific learning disability and mental illness   
e) multiple disabilities from amongst persons under clauses (a) to (d),   
f) other 'specified disabilities' mentioned in 'The Schedule' of the RPwD Act 2016.   
   
For the purpose of being considered for reservation, the applicable Central Government list as on the last 
date of CAT registration shall be binding. No subsequent changes will be effective for CAT 2026 and 
any subsequent selection process of the IIMs.   
The candidates belonging to the reserved categories need to also note the eligibility requirements carefully
------------------------------------------------------------
----- Document 2 -----
Source

### Task 2 

Create a table with the following columns: 

| Question | Retrieved Documents | Correct Answer? | Hallucination? | Faithful? | 

Fill the table for at least 10 test questions. 

### Task 3 

Intentionally ask a question that is not present in the uploaded documents. 

Observe whether: 
- The chatbot says "I don't know." 
- Or whether it hallucinates. 

| Question | Retrieved Documents | Correct Answer? | Hallucination? | Faithful? |
|----------|---------------------|-----------------|----------------|-----------|
| What are the eligibility criteria for CAT? | Eligibility PDF, Information Bulletin, Registration Guide | Yes (Partially) | No | Yes |
| When does CAT 2026 registration start? | Press Release, Information Bulletin, Registration Guide | Yes | No | Yes |
| When does CAT 2026 registration close? | Press Release, Information Bulletin | Yes | No | Yes |
| Which IIM is conducting CAT 2026? | Press Release, Registration Guide | No | No | Yes |
| What was the previous CAT cutoff for IIM Bangalore? (The information was not exist in the uploaded documents) | Information Bulletin, Press Release | Yes ("I don't know") | No | Yes |
| What are the mandatory documents required during registration? | Registration Guide, Information Bulletin | Yes  | No | Yes |
| What is the duration of the exam? | Press Release | Yes | No | Yes |
| How many test cities can a candidate select during registration? | Information Bulletin, Registration Guide | Yes | No | Yes |
| When will CAT 2026 results be declared? | Information Bulletin | Yes | No | Yes |
| What is the registration fee? | Information Bulletin, Press Release | Yes | No | Yes |


### Task 4 

Compare retrieval using: 
- Top-1 result 
- Top-3 results 
- Top-5 results 

Write your observations on answer quality.

In [9]:
# k=1
retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 1,
        "fetch_k": 20
    }
)
from langchain_ollama import OllamaLLM
#Load llm
llm = OllamaLLM(model="llama3.2")

# List of questions
questions = [
    "What is the eligibility criteria for CAT?",
    "When does CAT 2026 registration start?",
    "When does CAT 2026 registration close?",
    "Which IIM is conducting CAT 2026?",
    "What was the previous CAT cutoff for IIM Bangalore",
    "What are the mandatory documents required during registration?",
    "What is the duration of exam",
    "How many test cities can a candidate select during registration?",
    "When will CAT 2026 results be declared?",
    "What is the registration fee?"
]

# Loop through all questions
for question in questions:

    print("="*100)
    print("Question:")
    print(question)

    # Retrieve documents
    docs = retriever.invoke(question)

    print("\nRetrieved Documents:\n")

    context = ""

    for i, doc in enumerate(docs, start=1):

        print(f"----- Document {i} -----")
        print("Source:", doc.metadata["source"])
        print("Page:", doc.metadata["page"] + 1)
        print(doc.page_content)
        print("-"*60)

        context += doc.page_content + "\n\n"
    print(context)

    # Prompt
    prompt = f"""
You are a helpful assistant.

Answer ONLY using the information in the context below.

If the answer is not available in the context, reply:
"I don't know. The uploaded documents do not contain this information."

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer
    answer = llm.invoke(prompt)

    print("\nAnswer:\n")
    print(answer)
    print("\n")

Question:
What is the eligibility criteria for CAT?

Retrieved Documents:

----- Document 1 -----
Source: Documents\CAT_2026_Information_Bulletin_26-07-26.pdf
Page: 4
c) locomotor disability including cerebral palsy, leprosy cured, dwarfism, acid attack victims and 
muscular dystrophy,   
d) autism, intellectual disability, specific learning disability and mental illness   
e) multiple disabilities from amongst persons under clauses (a) to (d),   
f) other 'specified disabilities' mentioned in 'The Schedule' of the RPwD Act 2016.   
   
For the purpose of being considered for reservation, the applicable Central Government list as on the last 
date of CAT registration shall be binding. No subsequent changes will be effective for CAT 2026 and 
any subsequent selection process of the IIMs.   
The candidates belonging to the reserved categories need to also note the eligibility requirements carefully
------------------------------------------------------------
c) locomotor disability inclu

In [ ]:
# k=3
retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 20
    }
)
from langchain_ollama import OllamaLLM
#Load llm
llm = OllamaLLM(model="llama3.2")

# List of questions
questions = [
    "What is the eligibility criteria for CAT?",
    "When does CAT 2026 registration start?",
    "When does CAT 2026 registration close?",
    "Which IIM is conducting CAT 2026?",
    "What was the previous CAT cutoff for IIM Bangalore",
    "What are the mandatory documents required during registration?",
    "What is the duration of exam",
    "How many test cities can a candidate select during registration?",
    "When will CAT 2026 results be declared?",
    "What is the registration fee?"
]

# Loop through all questions
for question in questions:

    print("="*100)
    print("Question:")
    print(question)

    # Retrieve documents
    docs = retriever.invoke(question)

    print("\nRetrieved Documents:\n")

    context = ""

    for i, doc in enumerate(docs, start=1):

        print(f"----- Document {i} -----")
        print("Source:", doc.metadata["source"])
        print("Page:", doc.metadata["page"] + 1)
        print(doc.page_content)
        print("-"*60)

        context += doc.page_content + "\n\n"
    print(context)

    # Prompt
    prompt = f"""
You are a helpful assistant.

Answer ONLY using the information in the context below.

If the answer is not available in the context, reply:
"I don't know. The uploaded documents do not contain this information."

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer
    answer = llm.invoke(prompt)

    print("\nAnswer:\n")
    print(answer)
    print("\n")

Question:
What is the eligibility criteria for CAT?

Retrieved Documents:

----- Document 1 -----
Source: Documents\CAT_2026_Information_Bulletin_26-07-26.pdf
Page: 4
c) locomotor disability including cerebral palsy, leprosy cured, dwarfism, acid attack victims and 
muscular dystrophy,   
d) autism, intellectual disability, specific learning disability and mental illness   
e) multiple disabilities from amongst persons under clauses (a) to (d),   
f) other 'specified disabilities' mentioned in 'The Schedule' of the RPwD Act 2016.   
   
For the purpose of being considered for reservation, the applicable Central Government list as on the last 
date of CAT registration shall be binding. No subsequent changes will be effective for CAT 2026 and 
any subsequent selection process of the IIMs.   
The candidates belonging to the reserved categories need to also note the eligibility requirements carefully
------------------------------------------------------------
----- Document 2 -----
Source

| Retrieval | Observations |
|-----------|--------------|
| **Top-1 (k=1)** | Fastest retrieval but often missed important context. |
| **Top-3 (k=3)** | Provided a good balance between retrieval quality and efficiency. |
| **Top-5 (k=5)** | Delivered the best retrieval coverage by retrieving multiple relevant documents. |

### Task 5 

Research one RAG evaluation framework (such as Ragas or DeepEval) and 
summarize: 
- Features 
- Advantages 
- Use cases 

### **Ragas**

##### Features
- Evaluates RAG system performance automatically.
- Measures Faithfulness, Answer Relevancy, Context Precision, and Context Recall.
- Works with frameworks like LangChain and LlamaIndex.

##### Advantages
- Easy to use and open source.
- Reduces manual evaluation.
- Helps improve the accuracy of RAG applications.

##### Use Cases
- Testing RAG chatbots.
- Comparing different retrieval methods.
- Improving knowledge base quality.
- Monitoring RAG system performance.